# Data Modeling for Analytics

Notebook for **Chapter 2** of the *de* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/de/ch02.html

Tip: change a number, predict the output, then run the cell again.

## 2.1 Facts, Dimensions, Grain, Star and Snowflake

## 2.2 Slowly Changing Dimensions (SCD)

## 2.3 Inmon, Data Vault, and Choosing a Model

## 2.4 Modeling for Experts: Bridges, Late Data, Accumulating Snapshots and Data Vault

The star schema of 2.1 handles most reports. Real businesses keep breaking its assumptions: one sale belongs to *several* promotions, an order arrives before its customer exists, a process has milestones over days, and the source systems change without warning. This section gives you the standard answers, each with a run you can check.

In [ ]:
# Many-to-many with a bridge table: an order line can belong to several promotions.
import sqlite3
db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE fact_sales(sale_id INTEGER PRIMARY KEY, product TEXT, amount REAL, promo_group_key INTEGER);
CREATE TABLE dim_promo(promo_key INTEGER PRIMARY KEY, name TEXT);
CREATE TABLE bridge_promo(promo_group_key INTEGER, promo_key INTEGER, weight REAL);
INSERT INTO fact_sales VALUES (1,'phone',1000,10),(2,'case',100,20);
INSERT INTO dim_promo VALUES (1,'Diwali'),(2,'Bank offer'),(3,'Clearance');
INSERT INTO bridge_promo VALUES (10,1,0.5),(10,2,0.5),(20,3,1.0);
""")
naive = db.execute("""SELECT SUM(f.amount) FROM fact_sales f JOIN bridge_promo b USING(promo_group_key)""").fetchone()[0]
print("naive total after joining bridge (double counts):", naive, "| true total: 1100")
for row in db.execute("""SELECT p.name, SUM(f.amount * b.weight)
    FROM fact_sales f JOIN bridge_promo b USING(promo_group_key) JOIN dim_promo p USING(promo_key)
    GROUP BY p.name ORDER BY p.name"""): print(row)
print("weighted total:", db.execute("""SELECT SUM(f.amount*b.weight) FROM fact_sales f JOIN bridge_promo b USING(promo_group_key)""").fetchone()[0])

naive total after joining bridge (double counts): 2100.0 | true total: 1100
('Bank offer', 500.0)
('Clearance', 100.0)
('Diwali', 500.0)
weighted total: 1100.0


In [ ]:
# Late-arriving dimension (an "inferred member") and an accumulating snapshot fact.
import sqlite3
db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE dim_customer(customer_key INTEGER PRIMARY KEY AUTOINCREMENT, customer_id TEXT UNIQUE, name TEXT, is_inferred INTEGER);
CREATE TABLE fact_order(order_id TEXT PRIMARY KEY, customer_key INTEGER, placed TEXT, shipped TEXT, delivered TEXT);
""")
def key_for(cid):
    r = db.execute("SELECT customer_key FROM dim_customer WHERE customer_id=?", (cid,)).fetchone()
    if r: return r[0]
    db.execute("INSERT INTO dim_customer(customer_id,name,is_inferred) VALUES (?, 'Unknown', 1)", (cid,))
    return db.execute("SELECT customer_key FROM dim_customer WHERE customer_id=?", (cid,)).fetchone()[0]
# the order arrives BEFORE the customer record
db.execute("INSERT INTO fact_order(order_id,customer_key,placed) VALUES ('o1', ?, '2025-01-01')", (key_for("c9"),))
print("1", db.execute("SELECT * FROM dim_customer").fetchall())
# later the real customer row arrives: update the inferred row in place, the fact keeps its key
db.execute("UPDATE dim_customer SET name='Meera', is_inferred=0 WHERE customer_id='c9'")
print("2", db.execute("SELECT f.order_id, d.name FROM fact_order f JOIN dim_customer d USING(customer_key)").fetchall())
# accumulating snapshot: ONE row per order, milestones filled in as they happen
db.execute("UPDATE fact_order SET shipped='2025-01-03' WHERE order_id='o1'")
db.execute("UPDATE fact_order SET delivered='2025-01-06' WHERE order_id='o1'")
r = db.execute("""SELECT order_id, julianday(shipped)-julianday(placed), julianday(delivered)-julianday(shipped)
                  FROM fact_order""").fetchone()
print("3 order", r[0], "days to ship", r[1], "days in transit", r[2])

1 [(1, 'c9', 'Unknown', 1)]
2 [('o1', 'Meera')]
3 order o1 days to ship 2.0 days in transit 3.0


In [ ]:
# Data Vault style loading with hash keys: deterministic keys let hubs, links, satellites load in parallel.
import hashlib, sqlite3
def hk(*parts): return hashlib.md5("||".join(p.strip().upper() for p in parts).encode()).hexdigest()[:8]
db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE hub_customer(hk TEXT PRIMARY KEY, customer_id TEXT, load_ts TEXT);
CREATE TABLE hub_product(hk TEXT PRIMARY KEY, product_id TEXT, load_ts TEXT);
CREATE TABLE link_order(hk TEXT PRIMARY KEY, hk_customer TEXT, hk_product TEXT, load_ts TEXT);
CREATE TABLE sat_customer(hk TEXT, name TEXT, city TEXT, hashdiff TEXT, load_ts TEXT, PRIMARY KEY(hk, load_ts));
""")
def load(day, c_id, name, city, p_id):
    db.execute("INSERT OR IGNORE INTO hub_customer VALUES (?,?,?)", (hk(c_id), c_id, day))
    db.execute("INSERT OR IGNORE INTO hub_product VALUES (?,?,?)", (hk(p_id), p_id, day))
    db.execute("INSERT OR IGNORE INTO link_order VALUES (?,?,?,?)", (hk(c_id, p_id), hk(c_id), hk(p_id), day))
    diff = hk(name, city)
    last = db.execute("SELECT hashdiff FROM sat_customer WHERE hk=? ORDER BY load_ts DESC LIMIT 1", (hk(c_id),)).fetchone()
    if not last or last[0] != diff:                       # only store a new satellite row when attributes change
        db.execute("INSERT INTO sat_customer VALUES (?,?,?,?,?)", (hk(c_id), name, city, diff, day))
load("d1", "c1", "Asha", "Pune", "p1")
load("d2", "c1", "Asha", "Pune", "p1")       # replay of the same facts: nothing new
load("d3", "c1", "Asha", "Delhi", "p2")      # she moved and bought something else
for t in ("hub_customer", "hub_product", "link_order", "sat_customer"):
    print(t, db.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0])
print([r for r in db.execute("SELECT name, city, load_ts FROM sat_customer ORDER BY load_ts")])

hub_customer 1
hub_product 2
link_order 2
sat_customer 2
[('Asha', 'Pune', 'd1'), ('Asha', 'Delhi', 'd3')]


## Chapter recap

## Practice set

**1.** (easy) What is the grain of a table with one row per customer per month-end balance? **Answer:** one customer at one month end (a periodic snapshot).

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/de/ch02.html